In [1]:
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from viz_style import apply_style
apply_style()

import MixedEffectsModeling.config as config
from MixedEffectsModeling.core.model_engine_mixed import NormativeModelEngineMixed
from MixedEffectsModeling.core.ood_filter import MahalanobisFilter
from MixedEffectsModeling.core.shash import load_shash_params, shash_correct_matrix
from MixedEffectsModeling.validation.lobo_engine import load_full_data

ZDIR = Path('./Z_scores_mixed')
ZDIR.mkdir(parents=True, exist_ok=True)

In [2]:
# 0. Load production engine + disease covariates/counts (non-HC rows of the same QC-filtered cohort LOBO uses)
engine = NormativeModelEngineMixed.load(config.ENGINE_MIXED_DIR)
gene_names = [g for g in engine.genes if engine.genes[g].ok]

data = load_full_data()
dis_idx = np.where(~data['is_hc'])[0]
X_dis = data['X_raw'][dis_idx]
Y_dis = data['Y'][dis_idx][:, [data['gene_col'][g] for g in gene_names]]
names_dis = data['names'][dis_idx]
batch_dis = data['batch'][dis_idx]
phenotype_dis = data['phenotype'][dis_idx]

print(f"engine genes (ok): {len(gene_names)}")
print(f"disease samples: {len(dis_idx)}")
print(pd.Series(phenotype_dis).value_counts())

engine genes (ok): 19858
disease samples: 913
CAD_HF+               116
CAD_HF-               108
Tuberculosis          103
ME/CFS                 90
Pancreatitis           81
Pancreatic Cancer      74
Pre-eclampsia          62
Liver Cancer           48
Colorectal Cancer      41
Lung Cancer            33
Stomach Cancer         29
Esophagus Cancer       27
MM                     18
Other Cancer           18
HIV                    13
HIV + Tuberculosis     11
ICI-m                  11
ICI-treated Cancer     11
MGUS                    8
Pancreatic Cancer       6
Liver Cirrhosis         5
Name: count, dtype: int64


In [4]:
hc_train_mask = data['is_hc'] & ~np.isin(data['batch'], list(data['small_hc_batches']))
X_hc = data['X_raw'][hc_train_mask]
ood = MahalanobisFilter(percentile=95).fit(X_hc)
mahal_dist = ood.distances(X_dis)
ood_keep = mahal_dist <= ood.threshold_

summary = (pd.DataFrame({'phenotype': phenotype_dis, 'keep': ood_keep})
          .groupby('phenotype')['keep']
          .agg(n_total='size', n_kept='sum')
          .assign(pct_removed=lambda d: (1 - d['n_kept'] / d['n_total']) * 100)
          .sort_values('pct_removed', ascending=False))
print(f"Mahalanobis P95 threshold: {ood.threshold_:.2f}")
print(f"overall: kept {ood_keep.sum()}/{len(ood_keep)} ({(~ood_keep).mean()*100:.1f}% flagged OOD)")
print(summary.to_string())

Mahalanobis P95 threshold: 5.21
overall: kept 863/913 (5.5% flagged OOD)
                    n_total  n_kept  pct_removed
phenotype                                       
Pancreatic Cancer         6       2    66.666667
Liver Cirrhosis           5       4    20.000000
Stomach Cancer           29      24    17.241379
Liver Cancer             48      40    16.666667
Other Cancer             18      16    11.111111
Colorectal Cancer        41      37     9.756098
Lung Cancer              33      30     9.090909
Esophagus Cancer         27      25     7.407407
CAD_HF-                 108     101     6.481481
MM                       18      17     5.555556
Pre-eclampsia            62      59     4.838710
CAD_HF+                 116     112     3.448276
Pancreatic Cancer        74      72     2.702703
Pancreatitis             81      79     2.469136
Tuberculosis            103     101     1.941748
HIV                      13      13     0.000000
MGUS                      8       8     0.000

In [5]:
# 1. Score (raw RQR) -- cache-first
Z_PATH = ZDIR / 'Z_disease.npy'
GENES_PATH = ZDIR / 'gene_names.pkl'
META_PATH = ZDIR / 'sample_meta.csv'

if Z_PATH.exists():
    Z = np.load(Z_PATH)
else:
    Z = engine.score(X_dis, Y_dis, gene_names=gene_names)
    np.save(Z_PATH, Z)
    with open(GENES_PATH, 'wb') as f:
        pickle.dump(gene_names, f)

pd.DataFrame({'sample': names_dis, 'batch': batch_dis, 'phenotype': phenotype_dis,
             'ood_keep': ood_keep, 'mahal_dist': mahal_dist}).to_csv(META_PATH, index=False)

print(f"Z shape: {Z.shape}, finite frac: {np.isfinite(Z).mean():.4f}")

Z shape: (913, 19858), finite frac: 1.0000


In [6]:
# 2. SHASH-correct -- cache-first
Z_SHASH_PATH = ZDIR / 'Z_disease_shash.npy'

if Z_SHASH_PATH.exists():
    Z_shash = np.load(Z_SHASH_PATH)
else:
    shash_params = load_shash_params(config.CV_MIXED_DIR / 'cv_stats.csv')
    Z_shash = shash_correct_matrix(np.nan_to_num(Z, nan=0.0, posinf=10.0, neginf=-10.0), gene_names, shash_params)
    Z_shash[~np.isfinite(Z)] = np.nan
    np.save(Z_SHASH_PATH, Z_shash)

print(f"Z_shash shape: {Z_shash.shape}")
print(f"mean |Z| raw vs shash: {np.nanmean(np.abs(Z)):.4f} vs {np.nanmean(np.abs(Z_shash)):.4f}")

Z_shash shape: (913, 19858)
mean |Z| raw vs shash: 0.8521 vs 0.8542
